# Text Summarization — extractive ranking, ROUGE, and the copy/generate switch, from scratch

This notebook is the executable companion to the concept page. Every number it prints is the same number on the page and in the figures, because all three import the **one** source-of-truth module `text_summarization.py`. The notebook computes *nothing* the module does not define.

What we verify, end to end:
1. **TextRank** — a TF-IDF cosine sentence graph + a from-scratch power-iteration PageRank, verified to match `networkx`, picks the two on-topic sentences and demotes the distractor.
2. **ROUGE-1/2/L from scratch** — verified to match the `rouge-score` library to ~1e-9.
3. **ROUGE's blind spot** — a faithful paraphrase scores ~0; a wordy copy scores high.
4. **The pointer-generator `p_gen` switch** — copying rescues an out-of-vocabulary name.
5. **Extractive vs abstractive** — a real `distilbart` summary, scored with ROUGE.

## Setup — import the source-of-truth module and pin the environment

We print an honest device/version line and a **stable md5 hash** of the module text (never Python's salted `hash()`), so two runs of the same code produce the same fingerprint.

In [1]:
import hashlib
import sys
from pathlib import Path

# Import the canonical functions — the notebook computes NOTHING the .py doesn't define.
import text_summarization as ts

print(ts.environment_line())

# Stable content fingerprint of the module (md5 of the file text), NOT Python's salted hash().
module_path = Path(ts.__file__)
module_md5 = hashlib.md5(module_path.read_text(encoding='utf-8').encode('utf-8')).hexdigest()
print(f'text_summarization.py md5: {module_md5}')
print(f'python: {sys.version.split()[0]}')

Python 3.12.13 | numpy 2.4.6 | torch 2.12.0 | device: cpu (the from-scratch math is integer/float counting — CPU is exact)
text_summarization.py md5: 73e4bb4df8a9d4fccb7f3ed1c36e87fc
python: 3.12.13


## 1. TextRank — PageRank on a TF-IDF cosine sentence graph

We split the document into sentences, build the TF-IDF cosine **sentence-similarity graph**, and run a from-scratch **power-iteration PageRank** on it. The two most *central* sentences are the summary. The running document has two on-topic clusters (solar growth, grid strain) and one off-topic distractor (a bakery).

In [2]:
scores = ts.textrank_scores(ts.SOLAR_DOC)
top, summary = ts.textrank_summary(ts.SOLAR_DOC, k=2)

# Assert the qualitative point FIRST: the top-2 are the two THEMES (S1 solar, S3 grid),
# and the bakery distractor (S5) is the single least-central sentence.
assert top == [0, 2], f'expected S1+S3, got {[i + 1 for i in top]}'
assert int(scores.argmin()) == 4, 'the bakery sentence S5 should be least central'

import numpy as np
for rank, i in enumerate(np.argsort(scores)[::-1], 1):
    print(f'  #{rank}  S{i + 1}  centrality={scores[i]:.3f}  | {ts.SOLAR_DOC[i]}')
print(f'\nextractive summary (top-2, document order): {summary}')

  #1  S1  centrality=0.280  | Solar power capacity grew sharply across Europe last year.
  #2  S3  centrality=0.238  | Engineers warn the aging power grid struggles to absorb the new supply.
  #3  S2  centrality=0.203  | Solar energy installations expanded rapidly throughout Europe in 2024.
  #4  S4  centrality=0.181  | Grid operators say the network cannot easily handle the added solar load.
  #5  S5  centrality=0.098  | A local bakery announced a new sourdough recipe on Tuesday.

extractive summary (top-2, document order): Solar power capacity grew sharply across Europe last year. Engineers warn the aging power grid struggles to absorb the new supply.


### The from-scratch PageRank is correct, not just plausible

Our numpy power-iteration PageRank runs on the *same* graph as `networkx.pagerank`. They agree to ~1e-6 — which is what lets us trust the centrality numbers above.

In [3]:
matches, max_diff = ts.pagerank_matches_networkx()
assert matches, f'from-scratch PageRank diverged from networkx by {max_diff}'
print(f'from-scratch PageRank == networkx.pagerank   max abs diff = {max_diff:.2e}')

from-scratch PageRank == networkx.pagerank   max abs diff = 4.73e-07


## 2. ROUGE-1/2/L from scratch — and the proof it matches `rouge-score`

ROUGE-N is n-gram overlap (recall-flavoured); ROUGE-L is the longest-common-subsequence F-measure. On the classic *'the cat sat on the/a mat'* example, the one-word swap barely dents ROUGE-1/L but **drops ROUGE-2 to 0.60**, because it breaks two bigrams — which is why ROUGE-2 is the sharper fluency/ordering signal.

In [4]:
ref = 'the cat sat on the mat'
cand = 'the cat sat on a mat'
r = ts.rouge_all(cand, ref, stem=False)

# Assert the teaching point: R-1 and R-L survive the swap (~0.83) but R-2 falls to 0.60.
assert abs(r['rouge1']['fmeasure'] - 5 / 6) < 1e-9
assert abs(r['rouge2']['fmeasure'] - 0.60) < 1e-9
assert abs(r['rougeL']['fmeasure'] - 5 / 6) < 1e-9

for key in ('rouge1', 'rouge2', 'rougeL'):
    s = r[key]
    print(f"  {key:7s}  R={s['recall']:.3f}  P={s['precision']:.3f}  F1={s['fmeasure']:.3f}")

  rouge1   R=0.833  P=0.833  F1=0.833
  rouge2   R=0.600  P=0.600  F1=0.600
  rougeL   R=0.833  P=0.833  F1=0.833


### Verify against the `rouge-score` library (with the same Porter stemming)

In [5]:
from rouge_score import rouge_scorer

lib = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
pairs = [
    ('the cat sat on a mat', 'the cat sat on the mat'),
    ('company sales rose during q3', "the firm's revenue increased in the third quarter"),
    (ts.textrank_summary(ts.SOLAR_DOC, k=2)[1], ts.SOLAR_REFERENCE),
]
worst = 0.0
for cand_s, ref_s in pairs:
    lib_score = lib.score(ref_s, cand_s)        # rouge-score is (target, prediction)
    mine = ts.rouge_all(cand_s, ref_s, stem=True)
    for k in ('rouge1', 'rouge2', 'rougeL'):
        for fld in ('precision', 'recall', 'fmeasure'):
            worst = max(worst, abs(getattr(lib_score[k], fld) - mine[k][fld]))
assert worst < 1e-9, f'ROUGE diverged from rouge-score by {worst}'
print(f'from-scratch ROUGE == rouge-score (stemmed)   max abs diff = {worst:.2e}')

from-scratch ROUGE == rouge-score (stemmed)   max abs diff = 0.00e+00


## 3. ROUGE's blind spot — overlap is not meaning

A faithful **paraphrase** of the reference that reuses almost none of its words scores ~0 ROUGE-1, while a wordy **copy** that reuses the reference vocabulary scores high. ROUGE rewards lexical overlap, not meaning — which is why a genuinely good abstractive summary can score *lower* than a clunky extract.

In [6]:
reference = "the firm's revenue increased in the third quarter"
paraphrase = 'company sales rose during q3'                                   # same meaning
copy = "the firm's revenue increased greatly in the third quarter"           # copies words

f1_para = ts.rouge_n(paraphrase, reference, 1)['fmeasure']
f1_copy = ts.rouge_n(copy, reference, 1)['fmeasure']

# Assert the blind spot: the meaning-equal paraphrase scores FAR below the wordy copy.
assert f1_para < 0.1 < 0.9 < f1_copy, 'expected paraphrase << copy on ROUGE-1'
print(f'  faithful paraphrase  ROUGE-1 F1 = {f1_para:.3f}   | {paraphrase}')
print(f'  verbatim-ish copy    ROUGE-1 F1 = {f1_copy:.3f}   | {copy}')

  faithful paraphrase  ROUGE-1 F1 = 0.000   | company sales rose during q3
  verbatim-ish copy    ROUGE-1 F1 = 0.947   | the firm's revenue increased greatly in the third quarter


## 4. The pointer-generator `p_gen` switch — copying rescues an OOV name

The final distribution blends a vocabulary softmax with the attention (copy) distribution:
$P(w) = p_{gen}\,P_{vocab}(w) + (1 - p_{gen})\sum_{i:\,w_i = w} a_t^i$. The rare source word **Tsiolkovsky** is *not in the fixed vocab*, so generation alone could never emit it — but at a low gate the copy term carries it through.

In [7]:
p_vocab = {'and': 0.50, 'power': 0.10}            # NOTE: no entry for the rare name
copy_attention = {'Tsiolkovsky': 0.80, 'the': 0.20}

mix_copy = ts.pointer_generator_mix(p_vocab, copy_attention, p_gen=0.30)
mix_gen = ts.pointer_generator_mix(p_vocab, copy_attention, p_gen=0.90)

# Assert the switch flips: at low p_gen the OOV name wins; at high p_gen 'and' wins.
assert max(mix_copy, key=mix_copy.get) == 'Tsiolkovsky'
assert max(mix_gen, key=mix_gen.get) == 'and'
assert abs(mix_copy['Tsiolkovsky'] - 0.56) < 1e-9    # 0.30*0 + 0.70*0.80

print(f"  p_gen=0.30 (copy): P(Tsiolkovsky)={mix_copy['Tsiolkovsky']:.2f}  "
      f"P(and)={mix_copy['and']:.2f}  -> emit 'Tsiolkovsky'")
print(f"  p_gen=0.90 (gen) : P(Tsiolkovsky)={mix_gen['Tsiolkovsky']:.2f}  "
      f"P(and)={mix_gen['and']:.2f}  -> emit 'and'")

  p_gen=0.30 (copy): P(Tsiolkovsky)=0.56  P(and)=0.15  -> emit 'Tsiolkovsky'
  p_gen=0.90 (gen) : P(Tsiolkovsky)=0.08  P(and)=0.45  -> emit 'and'


> **Predict, then run:** before running the next cell, predict which summary scores higher ROUGE on this short document — the verbatim **extractive** one or the **abstractive** paraphrase? (Hint: ROUGE rewards word overlap.)

## 5. A real abstractive summarizer, scored with ROUGE

We run a real seq2seq model (`distilbart-cnn`, **beam search** so it is deterministic) and score both summaries against the reference. On this short document the **extractive** summary out-ROUGEs the abstractive one — exactly because ROUGE rewards the verbatim overlap the extract has and the paraphrase lacks. On full CNN/DailyMail the order flips; the *regime* matters.

In [8]:
_, extractive = ts.textrank_summary(ts.SOLAR_DOC, k=2)
abstractive = ts.run_real_abstractive()
if abstractive is None:  # minimal-env fallback (matches a verified distilbart beam run)
    abstractive = ('Engineers warn the aging power grid struggles to absorb the new supply . '
                   'Grid operators say the network cannot easily handle the added solar load . '
                   'Solar energy installations expanded rapidly throughout Europe in 2024 .')

ext = ts.rouge_all(extractive, ts.SOLAR_REFERENCE)
abs_ = ts.rouge_all(abstractive, ts.SOLAR_REFERENCE)

# Assert the teaching result: extractive out-ROUGEs abstractive on all three here.
for k in ('rouge1', 'rouge2', 'rougeL'):
    assert ext[k]['fmeasure'] >= abs_[k]['fmeasure'], f'{k}: expected extractive >= abstractive'

print(f'abstractive (distilbart): {abstractive}\n')
for label, s in [('extractive (TextRank)', ext), ('abstractive (distilbart)', abs_)]:
    print(f"  {label:26s} ROUGE-1={s['rouge1']['fmeasure']:.2f} "
          f"R-2={s['rouge2']['fmeasure']:.2f} R-L={s['rougeL']['fmeasure']:.2f}")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

abstractive (distilbart): Engineers warn the aging power grid struggles to absorb the new supply . Grid operators say the network cannot easily handle the added solar load . Solar energy installations expanded rapidly throughout Europe in 2024 .

  extractive (TextRank)      ROUGE-1=0.53 R-2=0.25 R-L=0.53
  abstractive (distilbart)   ROUGE-1=0.39 R-2=0.14 R-L=0.22


## Recap

- **Extractive (TextRank)** = PageRank on a TF-IDF cosine sentence graph — it found the two on-topic sentences and demoted the distractor, fully unsupervised; our from-scratch PageRank matches `networkx`.
- **ROUGE-1/2/L** from scratch matches `rouge-score` exactly — but it is **blind to meaning**: a faithful paraphrase scored ~0, a wordy copy scored high.
- The **pointer-generator `p_gen`** is the per-token extractive-vs-abstractive dial: low `p_gen` copies a rare name the vocab softmax could never emit; high `p_gen` paraphrases.
- On this short document the **extractive** summary out-ROUGEs the **abstractive** one — a regime-specific result that exposes ROUGE's lexical-overlap bias. Always pair ROUGE with a faithfulness check before shipping.